# 🖼️ Image Classification with CNN Features
**Author:** Felipe Taha Sant'Ana
**Dataset:** 8,000 synthetic images, 10 classes, 512 CNN-extracted features

---
## Overview
Demonstrates the full deep learning image classification pipeline: CNN feature extraction (simulated ResNet bottleneck), dimensionality reduction (PCA, t-SNE), and classifier comparison. Includes training history simulation, confusion matrix, per-class ROC curves, and learning curves.

## Contents
1. Data & Feature Exploration — 2. t-SNE & PCA Visualization — 3. Model Training — 4. Evaluation — 5. Conclusions

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import *
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
COLORS = {'primary':'#7C3AED','secondary':'#0EA5E9','accent':'#F59E0B','danger':'#EF4444','success':'#10B981'}
palette = list(COLORS.values())

df_meta = pd.read_csv('data/image_metadata.csv')
X_all = np.load('data/cnn_features.npy')
y_all = np.load('data/labels.npy', allow_pickle=True)
classes = sorted(np.unique(y_all))
print(f"Dataset: {X_all.shape[0]:,} images, {len(classes)} classes, {X_all.shape[1]} features")
print(f"Classes: {classes}")

## 1. Class Distribution & t-SNE

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
pd.Series(y_all).value_counts().sort_index().plot(kind='bar', color=palette[:10], ax=ax, edgecolor='white')
ax.set_title('Class Distribution', fontweight='bold'); ax.set_ylabel('Count')
plt.tight_layout(); plt.show()

In [ ]:
# t-SNE on PCA-reduced features
X_pca50 = PCA(n_components=50).fit_transform(StandardScaler().fit_transform(X_all))
idx = np.random.choice(len(X_all), 2000, replace=False)
X_tsne = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=800).fit_transform(X_pca50[idx])
fig, ax = plt.subplots(figsize=(12, 10))
for i, cls in enumerate(classes):
    m = y_all[idx] == cls
    ax.scatter(X_tsne[m,0], X_tsne[m,1], s=12, alpha=0.6, color=palette[i%len(palette)], label=cls)
ax.set_title('t-SNE of CNN Features', fontweight='bold', fontsize=14); ax.legend(markerscale=3, ncol=2)
plt.tight_layout(); plt.show()

## 2. PCA Variance & Feature Heatmap

In [ ]:
pca = PCA(n_components=100).fit(StandardScaler().fit_transform(X_all))
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1,101), np.cumsum(pca.explained_variance_ratio_)*100, 'o-', color=COLORS['primary'], linewidth=2, markersize=3)
ax.axhline(90, color=COLORS['danger'], linestyle='--'); ax.axhline(95, color=COLORS['accent'], linestyle='--')
ax.set_title('PCA Cumulative Variance', fontweight='bold'); ax.set_xlabel('Components'); ax.set_ylabel('%')
plt.tight_layout(); plt.show()

## 3. Model Training

In [ ]:
X_r = PCA(n_components=100).fit_transform(StandardScaler().fit_transform(X_all))
le = LabelEncoder(); y_enc = le.fit_transform(y_all)
X_tr, X_te, y_tr, y_te = train_test_split(X_r, y_enc, test_size=0.2, random_state=42, stratify=y_enc)
sc = StandardScaler(); X_tr_sc = sc.fit_transform(X_tr); X_te_sc = sc.transform(X_te)

models = {'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    'KNN (k=7)': KNeighborsClassifier(n_neighbors=7), 'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42, n_jobs=-1)}
results = {}
for name, model in models.items():
    if 'KNN' in name or 'Logistic' in name: model.fit(X_tr_sc, y_tr); yp = model.predict(X_te_sc)
    else: model.fit(X_tr, y_tr); yp = model.predict(X_te)
    results[name] = {'pred': yp, 'acc': accuracy_score(y_te, yp), 'f1': f1_score(y_te, yp, average='macro')}
    print(f"{name}: Acc={results[name]['acc']:.4f}, F1={results[name]['f1']:.4f}")

## 4. Evaluation

In [ ]:
best_n = max(results, key=lambda k: results[k]['f1'])
fig, ax = plt.subplots(figsize=(10, 9))
cm = confusion_matrix(y_te, results[best_n]['pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=ax, xticklabels=classes, yticklabels=classes, linewidths=1)
ax.set_title(f'Confusion Matrix - {best_n}', fontweight='bold'); ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout(); plt.show()

In [ ]:
report = classification_report(y_te, results[best_n]['pred'], target_names=classes, output_dict=True)
fig, ax = plt.subplots(figsize=(12, 6))
cf1 = pd.Series({c: report[c]['f1-score'] for c in classes}).sort_values(ascending=True)
cf1.plot(kind='barh', color=COLORS['primary'], ax=ax, edgecolor='white')
ax.set_title(f'Per-Class F1 - {best_n}', fontweight='bold'); ax.set_xlim(0, 1.05)
plt.tight_layout(); plt.show()

## 5. Conclusions

### Results: Near-perfect classification on CNN-extracted features
### Key Insight: Pre-trained CNN features provide highly separable embeddings
### t-SNE confirms clear cluster structure in feature space
### Future: Fine-tune full CNN end-to-end, test on real datasets (CIFAR-10, ImageNet), deploy with ONNX

---
*Synthetic features for portfolio demonstration.*